> 在线讲义：[第 37 章 · 从并行方案到超节点规模：按期限、成本和故障挑方案](https://wikieden.github.io/ai-infra-from-constraints/lessons/37-scale-choice.html) · 本书《从约束推导 AI Infra》

# 第 37 章 · 从并行方案到超节点规模：按期限、成本和故障挑方案

配套讲义：仓库里 `ai-infra/notes/lessons/37-scale-choice.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



四个 128K 会话续写 8 个 token 的部署比较、按时会话成本、故障、32K 对照取自 AII 第 6 章 §6.7.1–§6.7.3。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
KiB, MiB = 2**10, 2**20
HBM = 3350e9

**函数 `step_bytes`**：式 (6-4)（同第 32 章）：Qwen3-32B 一步的 HBM 访问量。

In [ ]:
def step_bytes(s: int) -> float:
    """式 (6-4)（同第 32 章）：Qwen3-32B 一步的 HBM 访问量。"""
    return 64 * 930 * MiB + 256 * KiB * (s + 1) + 151936 * 5120 * 2

**函数 `ring_ar`**：式 (6-9)（同第 35 章）：环形 AllReduce。

In [ ]:
def ring_ar(n: int, M: float, alpha: float = 0.822e-6, bw: float = 450e9) -> float:
    """式 (6-9)（同第 35 章）：环形 AllReduce。"""
    return 2 * (n - 1) * alpha + 2 * (n - 1) * M / (n * bw) if n > 1 else 0.0

**函数 `eight_steps_ms`**：一个会话续写 8 个 token 的时间（式 6-4、6-5）。

In [ ]:
def eight_steps_ms(p: int, s0: int = 131064) -> float:
    """一个会话续写 8 个 token 的时间（式 6-4、6-5）。"""
    return sum(step_bytes(s) / HBM / p + 128 * ring_ar(p, 10 * KiB) for s in range(s0, s0 + 8)) * 1e3

**函数 `finish_times`**：小工具。

In [ ]:
def finish_times(p: int, sessions: int = 4, cards: int = 8, **kw) -> list[float]:
    t = eight_steps_ms(p, **kw)
    inst = cards // p
    return [t * (i // inst + 1) for i in range(sessions)]

**函数 `cost_per_on_time`**：(占用 GPU·s, 按时会话数, 每个按时会话的 GPU·s)。

In [ ]:
def cost_per_on_time(finish: list[float], deadline: float, cards: int = 8) -> tuple[float, int, float]:
    """(占用 GPU·s, 按时会话数, 每个按时会话的 GPU·s)。"""
    occ = cards * max(finish) / 1e3
    n = sum(1 for f in finish if f <= deadline)
    return occ, n, occ / n if n else float("inf")

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：四个 128K 会话各续写 8 个 token，八张 H100")
for p in (2, 4, 8):
    f = finish_times(p)
    occ, n, c = cost_per_on_time(f, 130)
    print(f"  {8 // p} 个 TP{p}：单会话 {eight_steps_ms(p):.1f} ms，完成 {[round(x, 1) for x in f]}，"
          f"占用 {occ:.3f} GPU·s，按时 {n}，每个 {c:.3f}")

In [ ]:
print("\n第二步：20 ms 卡 0 故障，停 40 ms 后从头重做")
t2 = eight_steps_ms(2)
f = [60 + t2] + [t2] * 3
occ, n, c = cost_per_on_time(f, 130)
print(f"  4 个 TP2：完成 {[round(x, 1) for x in f]}，占用 {occ:.3f}，每个按时会话 {c:.3f} GPU·s")
print(f"  能耗（10.2 kW）：TP2 {10.2 * t2 / 1e3:.2f} kJ")

In [ ]:
print("\n第三步：上下文改成 32K")
for p in (1, 2):
    print(f"  TP{p}：8 步 {eight_steps_ms(p, s0=32760):.1f} ms")
f = finish_times(2, s0=32760)
print(f"  4 个 TP2 每个按时会话 {cost_per_on_time(f, 130)[2]:.3f} GPU·s")

## 3. 代码里才看得到的

1. 8 步里上下文从 131,064 涨到 131,071，每步 KV 多读一个 token，代码逐步相加而不是乘 8。
2. 32K 的起点取 32,760，使 8 步正好写满 32,768 个位置。
3. 故障那行只改受影响会话的完成时刻（60 + 119.1），其余实例不受影响。

## 4. 自己改一改

- 把 `finish_times` 改成按到达时间分配（会话 0、1 先到，2、3 晚 30 ms 到）。
- 给 `cost_per_on_time` 加能耗：每卡 1.275 kW。
- 把 `eight_steps_ms` 的 $T_{\mathrm{AR}}$ 换成树形算法，三种部署的排名变不变？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 八个厨师四桌客，两人一灶齐上菜，围一灶第一桌快、后面排队；
> 包车按时长算，摊给准时到的人，抛锚车费照收；
> 先列食材忌口，再按毛病选做法，写步骤比时间，最后试吃；
> 仓库扩建多接单，单个快递不更快；权重固定后看每跳。